---
title: "3.1 First plots from a DataFrame"
---

<div class="lesson-tools">
<a href="https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/03-viz/01-first-plots.ipynb" class="btn btn-primary">Open in Colab</a>
<a href="https://ningzhanglab.github.io/Python-for-Beginners/downloads/03-viz/01-first-plots.ipynb" class="btn btn-outline-secondary">Download notebook</a>
</div>

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [2.8 Dates and time series basics](../02-data/08-dates-time-series.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Choose line, bar and histogram plots for different questions.
- Create and label a figure from a pandas table.
- Keep dates, units, missingness and aggregation visible.

## Why it matters

A chart should answer a question you can state in a sentence. A line can show change over time, bars can compare groups, and a histogram can show a distribution—the spread of values across observations. We begin with made-up numbers for Northland, then plot Canada's historical observations and a regional summary. Plotting a table does not make its sample representative or its patterns causal.

## Worked example

### Meet the Figure and Axes

A **Figure** is the whole image; an **Axes** is a plotting area inside it. `plt.subplots` creates both. pandas' `.plot` can draw onto a supplied Axes, so you can keep table operations separate from labels and layout. This small table contains made-up life expectancies for fictional Northland. A marker shows each actual observed point.

In [ ]:
#| label: fig-fictional-line
#| fig-alt: "Line chart of made-up Northland life expectancy: 68 years in 2000, 71 in 2010 and 74 in 2020. Markers identify the three observations."
#| fig-cap: "Made-up numbers for a fictional country."
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12})
small = pd.DataFrame({"year": [2000, 2010, 2020], "life_exp": [68.0, 71.0, 74.0]})
fig_line, ax_line = plt.subplots(figsize=(6, 4), layout="constrained")
small.plot(x="year", y="life_exp", ax=ax_line, marker="o", legend=False, color="#21645a")
ax_line.set(xlabel="Year", ylabel="Life expectancy at birth (years)", title="Northland · made-up numbers")
plt.show()

The three points rise steadily. The connecting segments are visual guides; they do not establish values for unobserved years. `layout="constrained"` reserves space for labels. `legend=False` removes an unnecessary one-series legend while the y-axis label states the measurement.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Plot a real series in time order

Choose one country and sort by year before drawing a line. These are five-year observations, not annual measurements. Keep a separate name for the figure so later instructions cannot accidentally change a different chart.

In [ ]:
#| label: fig-canada-line
#| fig-alt: "Canada life expectancy rises from 68.75 years in 1952 to 80.653 in 2007 across twelve observed five-year points. Lines connect the observations."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise."
countries = pd.read_csv(data_file)
canada = countries.loc[countries["country"] == "Canada"].sort_values("year")
fig_canada, ax_canada = plt.subplots(figsize=(6, 4), layout="constrained")
canada.plot(x="year", y="life_exp", ax=ax_canada, marker="o", legend=False, color="#21645a")
ax_canada.set(xlabel="Observed year (five-year intervals)", ylabel="Life expectancy at birth (years)", title="Canada · 1952–2007")
plt.show()

The broad increase describes Canada in this historical extract. The graph does not identify which policy, income change or health improvement caused it. Sorting the full multi-country panel into one connected line would join unrelated countries.

### Compare regional summaries with bars

For a group comparison, select one year before calculating equal-country regional means. A bar starts from a common zero baseline so its length represents the displayed quantity. A mean life expectancy is not a count of people or an official regional life table.

In [ ]:
#| label: fig-region-bars
#| fig-alt: "Horizontal bars compare equal-country mean life expectancy across the five source regions in 2007. Africa has the lowest mean and Oceania the highest; the axis begins at zero."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise. Oceania contains only two countries in this extract."
year_2007 = countries.loc[countries["year"] == 2007].copy()
region_means = year_2007.groupby("region")["life_exp"].mean().sort_values()
fig_bar, ax_bar = plt.subplots(figsize=(6, 4), layout="constrained")
region_means.plot.barh(ax=ax_bar, color="#21645a")
ax_bar.set(xlabel="Mean country life expectancy (years)", ylabel="Source continent group", title="Equal-country means · 2007", xlim=(0, 90))
plt.show()

The ordering reflects this sample and year. The unequal number of countries per group matters: Africa has 52 while Oceania has two. Equal-country means differ from population-weighted averages. For a measured quantity, a dot plot can often show differences more compactly; bars are used here to introduce the command.

### Inspect the distribution with a histogram

A **histogram** groups a numeric measurement into intervals called bins and counts observations in each interval. Here one observation is one country in 2007. Bins change the display, so show the chosen count and try a second count in the exercise.

In [ ]:
#| label: fig-life-histogram
#| fig-alt: "Histogram of 142 country life expectancies in 2007 using twelve bins. Values extend from about 40 to 83 years, with many observations in the 70s."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise."
fig_hist, ax_hist = plt.subplots(figsize=(6, 4), layout="constrained")
year_2007["life_exp"].plot.hist(bins=12, ax=ax_hist, color="#21645a", edgecolor="white")
ax_hist.set(xlabel="Life expectancy at birth (years)", ylabel="Number of countries", title="Life expectancy distribution · 2007")
plt.show()

Each country contributes one count, regardless of its population. The histogram cannot show which countries are where; use a labelled table for exact identities. pandas omits missing values from a histogram, so inspect missingness before interpreting its count.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Plot a second country

Select Japan, sort its years and draw a labelled life-expectancy line with point markers.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
#| label: fig-exercise-japan
#| fig-alt: "Japan life expectancy rises from 63.03 years in 1952 to 82.603 in 2007 across twelve five-year observations."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise."
japan = countries.loc[countries["country"] == "Japan"].sort_values("year")
fig_japan, ax_japan = plt.subplots(figsize=(6, 4), layout="constrained")
japan.plot(x="year", y="life_exp", marker="o", legend=False, color="#21645a", ax=ax_japan)
ax_japan.set(xlabel="Observed year", ylabel="Life expectancy at birth (years)", title="Japan · 1952–2007")
plt.show()

Japan has twelve observations in this extract. Do not interpret the connecting line as evidence of annual measurements.

</details>

### 2. Change histogram bins

Make the 2007 life-expectancy histogram with six bins. Which details become less visible?

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
#| label: fig-exercise-bins
#| fig-alt: "The same 142 country observations shown with six wider life-expectancy bins rather than twelve."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise."
fig_six, ax_six = plt.subplots(figsize=(6, 4), layout="constrained")
year_2007["life_exp"].plot.hist(bins=6, ax=ax_six, color="#21645a", edgecolor="white")
ax_six.set(xlabel="Life expectancy at birth (years)", ylabel="Number of countries", title="Same 142 countries · six bins")
plt.show()

Wider bins smooth away detail without changing the underlying observations. A different bin rule is a different view, not a new dataset.

</details>

### 3. Report group counts

Print the number of country observations behind each 2007 regional mean. What limitation does this reveal?

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
region_counts = year_2007.groupby("region")["country"].size()
print(region_counts)

The counts are 52, 25, 33, 30 and 2 for Africa, Americas, Asia, Europe and Oceania. Bar height alone does not reveal those sample sizes; include them in a caption or companion table.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### A plot uses a missing column

The table's variable names must match the plot arguments.

In [ ]:
try:
    canada[["year", "life_expectancy"]].plot(x="year", y="life_expectancy")
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Use the actual life_exp field. An unfamiliar chart should prompt a check of the selected data and column meanings, not just a change of colour.

### A plotting keyword is misspelled

This intentionally requests an absent Axes method.

In [ ]:
try:
    ax_canada.set_y_label("Years")
except AttributeError as error:
    print(f"{type(error).__name__}: {error}")

The method is set_ylabel, with no underscore between y and label. The Figure and Axes have different operations; keep track of which object you are using.

## Recap

- Use a line for ordered time observations, bars for group comparisons and a histogram for a distribution.
- Label measurement units, time periods, weights and source coverage.
- Count the observations behind plotted summaries.
- Keep named Figure/Axes handles and inspect the data before adjusting the chart.

**Next:** [3.2 Making a chart readable](../03-viz/02-readable-charts.ipynb).